# HRCT Temporal Bone Pathology Detection - Full Pipeline Demo

This notebook demonstrates the complete pipeline for detecting temporal bone pathologies using 3D ResNet-18.

**Pathologies detected:**
1. Cholesteatoma (primary)
2. Ossicular chain discontinuity (secondary)

**Prerequisites:**
- All dependencies installed (see requirements.txt)
- ROI data extracted (Phase 1-2 complete)
- GPU recommended but not required

## 1. Setup & Imports

In [ ]:
# Installation (uncomment if needed)
# !pip install -r requirements.txt
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
import torch
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Dataset Overview

In [ ]:
# Load labels
labels_df = pd.read_csv('../labels.csv')
print(f"Total entries: {len(labels_df)}")
print(f"Patients: {labels_df['patient_id'].nunique()}")

# Filter to included samples
included = labels_df[labels_df['exclusion_status'] == 'include'].copy()
print(f"\nIncluded ears: {len(included)}")

# Display sample
included.head(10)

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Cholesteatoma
chole_counts = included['cholesteatoma'].value_counts().sort_index()
axes[0].bar(['Positive', 'Negative'], [chole_counts.get(1, 0), chole_counts.get(0, 0)], 
           color=['#2E86AB', '#E94F37'])
axes[0].set_title('Cholesteatoma Distribution', fontsize=14)
axes[0].set_ylabel('Count')

# Ossicular
ossic_counts = included['ossicular_discontinuity'].value_counts().sort_index()
axes[1].bar(['Positive', 'Negative'], [ossic_counts.get(1, 0), ossic_counts.get(0, 0)],
           color=['#2E86AB', '#E94F37'])
axes[1].set_title('Ossicular Discontinuity Distribution', fontsize=14)
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f"Cholesteatoma: {chole_counts.get(1, 0)} positive, {chole_counts.get(0, 0)} negative")
print(f"Ossicular: {ossic_counts.get(1, 0)} positive, {ossic_counts.get(0, 0)} negative")

## 3. Visualize Sample ROI

In [ ]:
# Load a sample ROI
roi_dir = Path('../roi_extracted')
sample_patients = list(roi_dir.iterdir())

if sample_patients:
    # Find first valid ROI
    sample_roi = None
    for patient_dir in sample_patients:
        for side in ['left', 'right']:
            roi_path = patient_dir / side / 'axial_roi.npy'
            if roi_path.exists():
                sample_roi = np.load(roi_path)
                print(f"Loaded ROI from: {roi_path}")
                break
        if sample_roi is not None:
            break

    if sample_roi is not None:
        print(f"ROI shape: {sample_roi.shape}")
        print(f"Value range: [{sample_roi.min():.2f}, {sample_roi.max():.2f}]")
        
        # Show 3 orthogonal views
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        center = [s // 2 for s in sample_roi.shape]
        
        axes[0].imshow(sample_roi[center[0], :, :], cmap='gray')
        axes[0].set_title(f'Axial (slice {center[0]})', fontsize=12)
        axes[0].axis('off')
        
        axes[1].imshow(sample_roi[:, center[1], :], cmap='gray')
        axes[1].set_title(f'Sagittal (slice {center[1]})', fontsize=12)
        axes[1].axis('off')
        
        axes[2].imshow(sample_roi[:, :, center[2]], cmap='gray')
        axes[2].set_title(f'Coronal (slice {center[2]})', fontsize=12)
        axes[2].axis('off')
        
        plt.suptitle('ROI Orthogonal Views', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
else:
    print("No ROI data found. Run Phase 1-2 first.")

## 4. Phase 3 - Dataset Stratification

Check existing splits or run stratification.

In [ ]:
# Check if splits already exist
split_dir = Path('dataset_splits_validation')

if split_dir.exists() and (split_dir / 'split_metadata.json').exists():
    with open(split_dir / 'split_metadata.json') as f:
        metadata = json.load(f)
    print("Existing splits found!")
    print(f"\nSplit Statistics:")
    print(f"  Total patients: {metadata.get('n_patients', 'N/A')}")
    print(f"  Total ears: {metadata.get('n_ears', 'N/A')}")
    print(f"  CV folds: {metadata.get('n_folds', 5)}")
    print(f"  Random seed: {metadata.get('random_seed', 'N/A')}")
else:
    print("No existing splits. Run Phase 3 script:")
    print("\n!python pipeline/phase3_dataset_stratification_validation.py --roi_dir roi_extracted --labels_csv labels.csv --output_dir dataset_splits_validation")

In [ ]:
# Visualize fold distributions
if split_dir.exists():
    fold_data = []
    for fold_file in sorted(split_dir.glob('fold_*.json')):
        with open(fold_file) as f:
            fold = json.load(f)
            fold_num = fold.get('fold', int(fold_file.stem.split('_')[1]))
            fold_data.append({
                'Fold': fold_num,
                'Train Patients': fold.get('n_train_patients', len(fold.get('train_patient_ids', []))),
                'Val Patients': fold.get('n_val_patients', len(fold.get('val_patient_ids', []))),
                'Train Ears': fold.get('n_train_ears', len(fold.get('train_ear_ids', []))),
                'Val Ears': fold.get('n_val_ears', len(fold.get('val_ear_ids', [])))
            })
    
    fold_df = pd.DataFrame(fold_data)
    display(fold_df)

## 5. Augmentation Visualization

In [ ]:
# Demonstrate augmentation effects (if MONAI available)
try:
    from monai.transforms import Compose, RandAffined, RandGaussianNoised, EnsureChannelFirstd
    
    if sample_roi is not None:
        # Define augmentation pipeline
        transform = Compose([
            EnsureChannelFirstd(keys=["image"]),
            RandAffined(
                keys=["image"],
                mode="bilinear",
                rotate_range=[0.26, 0.26, 0.26],  # ±15°
                translate_range=[10, 10, 10],
                scale_range=[0.05, 0.05, 0.05],
                prob=1.0  # Always apply for demo
            ),
        ])
        
        # Apply augmentation
        sample_dict = {"image": sample_roi}
        augmented = transform(sample_dict)["image"][0].numpy()  # Remove channel dim
        
        # Compare original vs augmented
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        center = sample_roi.shape[0] // 2
        
        # Original
        axes[0, 0].imshow(sample_roi[center, :, :], cmap='gray')
        axes[0, 0].set_title('Original - Axial')
        axes[0, 1].imshow(sample_roi[:, center, :], cmap='gray')
        axes[0, 1].set_title('Original - Sagittal')
        axes[0, 2].imshow(sample_roi[:, :, center], cmap='gray')
        axes[0, 2].set_title('Original - Coronal')
        
        # Augmented
        axes[1, 0].imshow(augmented[center, :, :], cmap='gray')
        axes[1, 0].set_title('Augmented - Axial')
        axes[1, 1].imshow(augmented[:, center, :], cmap='gray')
        axes[1, 1].set_title('Augmented - Sagittal')
        axes[1, 2].imshow(augmented[:, :, center], cmap='gray')
        axes[1, 2].set_title('Augmented - Coronal')
        
        for ax in axes.flat:
            ax.axis('off')
        
        plt.suptitle('Data Augmentation Example (Rotation + Translation + Scaling)', fontsize=14)
        plt.tight_layout()
        plt.show()
    else:
        print("Load a sample ROI first.")
except ImportError:
    print("MONAI not installed. Run: pip install monai")

## 6. Phase 4 - Training (Demo)

Run a quick training demo with reduced epochs.

In [ ]:
# Check if models already exist
models_dir = Path('models_validation')

if models_dir.exists():
    existing_folds = list(models_dir.glob('fold_*/best_checkpoint.pth'))
    print(f"Found {len(existing_folds)} trained folds:")
    for fold in existing_folds:
        print(f"  - {fold.parent.name}")
else:
    print("No trained models found.")
    print("\nTo train (this will take time), run:")
    print("!python pipeline/phase4_model_training_validation.py --split_dir dataset_splits_validation --roi_dir roi_extracted --labels_csv labels.csv --output_dir models_validation --fold 0 --epochs 10")

In [ ]:
# Visualize training history (if available)
history_files = list(models_dir.glob('fold_*/training_history.json')) if models_dir.exists() else []

if history_files:
    fig, axes = plt.subplots(len(history_files), 2, figsize=(14, 5 * len(history_files)))
    if len(history_files) == 1:
        axes = axes.reshape(1, -1)
    
    for idx, history_file in enumerate(history_files):
        with open(history_file) as f:
            history = json.load(f)
        
        fold_name = history_file.parent.name
        epochs = range(1, len(history.get('train_loss', [])) + 1)
        
        # Loss plot
        if 'train_loss' in history:
            axes[idx, 0].plot(epochs, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
        if 'val_loss' in history:
            axes[idx, 0].plot(epochs, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
        axes[idx, 0].set_xlabel('Epoch')
        axes[idx, 0].set_ylabel('Loss')
        axes[idx, 0].set_title(f'{fold_name} - Training Loss')
        axes[idx, 0].legend()
        axes[idx, 0].grid(True, alpha=0.3)
        
        # AUC plot
        auc_keys = [k for k in history.keys() if 'auc' in k.lower()]
        for key in auc_keys:
            label = key.replace('val_auc_', '').replace('_', ' ').title()
            axes[idx, 1].plot(epochs, history[key], label=label, linewidth=2)
        axes[idx, 1].set_xlabel('Epoch')
        axes[idx, 1].set_ylabel('AUC-ROC')
        axes[idx, 1].set_title(f'{fold_name} - Validation AUC')
        axes[idx, 1].legend()
        axes[idx, 1].set_ylim([0, 1])
        axes[idx, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No training history found. Train a model first.")

## 7. Phase 5 - Evaluation

In [ ]:
# Check for existing evaluation results
eval_dir = Path('evaluation_results_validation')

if eval_dir.exists() and (eval_dir / 'cv_aggregated_metrics.json').exists():
    print("Existing evaluation results found!")
    with open(eval_dir / 'cv_aggregated_metrics.json') as f:
        metrics = json.load(f)
    
    # Display results table
    agg = metrics.get('aggregated_metrics', {})
    if agg:
        results_data = []
        for pathology, m in agg.items():
            results_data.append({
                'Pathology': pathology.replace('_', ' ').title(),
                'AUC (mean ± std)': f"{m.get('auc_mean', 0):.3f} ± {m.get('auc_std', 0):.3f}",
                'Sensitivity': f"{m.get('sensitivity_mean', 0):.3f} ± {m.get('sensitivity_std', 0):.3f}" if m.get('sensitivity_mean') else 'N/A',
                'Specificity': f"{m.get('specificity_mean', 0):.3f} ± {m.get('specificity_std', 0):.3f}" if m.get('specificity_mean') else 'N/A',
                'N Folds': m.get('n_folds', 0)
            })
        
        results_df = pd.DataFrame(results_data)
        display(results_df)
else:
    print("No evaluation results found.")
    print("\nTo run evaluation:")
    print("!python pipeline/phase5_model_evaluation_validation.py --models_dir models_validation --split_dir dataset_splits_validation --roi_dir roi_extracted --labels_csv labels.csv --output_dir evaluation_results_validation")

In [ ]:
# Display evaluation plots if available
figures_dir = eval_dir / 'figures' if eval_dir.exists() else None

if figures_dir and figures_dir.exists():
    plot_files = list(figures_dir.glob('*.png'))
    
    if plot_files:
        n_plots = len(plot_files)
        fig, axes = plt.subplots(1, min(n_plots, 3), figsize=(6 * min(n_plots, 3), 5))
        if n_plots == 1:
            axes = [axes]
        
        for ax, plot_file in zip(axes, plot_files[:3]):
            img = plt.imread(plot_file)
            ax.imshow(img)
            ax.set_title(plot_file.stem.replace('_', ' ').title())
            ax.axis('off')
        
        plt.tight_layout()
        plt.show()
    else:
        print("No plots found in figures directory.")
else:
    print("Figures directory not found.")

## 8. Sanity Checks & Summary

In [ ]:
print("Pipeline Validation Checks:")
print("=" * 50)

# Load evaluation metrics if available
if eval_dir.exists() and (eval_dir / 'cv_aggregated_metrics.json').exists():
    with open(eval_dir / 'cv_aggregated_metrics.json') as f:
        metrics = json.load(f)
    
    sanity = metrics.get('sanity_checks', {})
    for check, passed in sanity.items():
        status = "✓ PASS" if passed else "✗ FAIL"
        print(f"{check.replace('_', ' ').title()}: [{status}]")
else:
    print("Run evaluation to see sanity checks.")

print("\n" + "=" * 50)
print("\nNext Steps:")
print("1. Collect remaining patients (target: 100-120 ears)")
print("2. Re-run full pipeline with production scripts")
print("3. Include facial nerve head (if sufficient documented cases)")
print("4. Evaluate on fixed test set")
print("5. Prepare manuscript")

---

## Quick Commands Reference

```bash
# Phase 3: Stratification
python pipeline/phase3_dataset_stratification_validation.py \
    --roi_dir roi_extracted \
    --labels_csv labels.csv \
    --output_dir dataset_splits_validation

# Phase 4: Training (single fold)
python pipeline/phase4_model_training_validation.py \
    --split_dir dataset_splits_validation \
    --roi_dir roi_extracted \
    --labels_csv labels.csv \
    --output_dir models_validation \
    --fold 0 --epochs 50

# Phase 5: Evaluation
python pipeline/phase5_model_evaluation_validation.py \
    --models_dir models_validation \
    --split_dir dataset_splits_validation \
    --roi_dir roi_extracted \
    --labels_csv labels.csv \
    --output_dir evaluation_results_validation
```